# 🔍 Vector Search Deep Dive — Hands-on PoC

> **교육 목적**: 각 주제의 핵심 개념을 Elasticsearch 클러스터에서 직접 실행하며 체득  
> **환경**: Elasticsearch 9.x (ECH), Python `elasticsearch` 클라이언트, GitHub Codespace  
> **표기**: 모든 쿼리는 **Query DSL** + **ES|QL** 병기  
> **ES|QL 버전 참고**: `FORK`/`FUSE`는 9.1+ tech preview, `RERANK`/`COMPLETION`은 9.2+ tech preview

---

## 📋 목차

| Part | 주제 | 핵심 PoC |
|------|------|----------|
| 0 | 검색의 진화 — Lexical vs Vector | BM25 한계 체험 → Vector 의미 검색 |
| 1 | 벡터 검색 내부 원리 — HNSW | HNSW 파라미터 튜닝 + num_candidates 효과 |
| 2 | ML 모델 분류 — Embedding · Reranker | Inference Endpoint 생성 + Bi/Cross-encoder 비교 |
| 3 | Chunking + semantic_text vs dense_vector | 자동 chunking vs 수동 제어 비교 |
| 4 | 양자화 — BBQ · DiskBBQ | float32 vs BBQ 압축률 + recall 비교 |
| 5 | Hybrid Search — RRF · Linear · Reranker | 3단계 Retriever + ES\|QL FORK/FUSE/RERANK |
| 6 | RAG 실전 설계 | Context Engineering + min_score cutoff |

## 🛠️ 환경 설정

`ES_CLOUD_ID`와 `ES_API_KEY` 환경변수가 설정되어 있어야 합니다.

```bash
# GitHub Codespace 터미널에서:
export ES_CLOUD_ID="your-deployment:..."
export ES_API_KEY="your-api-key"
```

In [ ]:
# 의존성 설치 (Codespace 첫 실행 시)
!pip install elasticsearch -q

In [ ]:
import os, json, time, random, math
from elasticsearch import Elasticsearch, helpers

# --- 연결 ---
ES_CLOUD_ID = os.environ["ES_CLOUD_ID"]
ES_API_KEY  = os.environ["ES_API_KEY"]

es = Elasticsearch(cloud_id=ES_CLOUD_ID, api_key=ES_API_KEY)

info = es.info()
print(f"✅ Connected: {info['cluster_name']} (v{info['version']['number']})")

In [ ]:
# ── 유틸리티 함수 ──────────────────────────────────────────

def dsl(method, path, body=None, **kwargs):
    """Query DSL 요청을 실행하고 결과를 포맷 출력"""
    resp = es.perform_request(method, path, body=body, **kwargs)
    print(json.dumps(resp.body if hasattr(resp, 'body') else resp, indent=2, ensure_ascii=False))
    return resp.body if hasattr(resp, 'body') else resp

def esql(query, **kwargs):
    """ES|QL 쿼리를 실행하고 테이블 형태로 출력"""
    resp = es.esql.query(query=query, format="json", **kwargs)
    body = resp.body if hasattr(resp, 'body') else resp
    cols = [c["name"] for c in body.get("columns", [])]
    rows = body.get("values", [])
    
    if not cols:
        print("(empty result)")
        return body
    
    # 컬럼 폭 계산
    widths = [max(len(str(c)), max((len(str(r[i])) for r in rows), default=0)) for i, c in enumerate(cols)]
    
    header = " | ".join(str(c).ljust(w) for c, w in zip(cols, widths))
    sep = "-+-".join("-" * w for w in widths)
    print(header)
    print(sep)
    for r in rows[:20]:  # 최대 20행
        print(" | ".join(str(v).ljust(w) for v, w in zip(r, widths)))
    if len(rows) > 20:
        print(f"... ({len(rows)} rows total)")
    
    return body

def cleanup_indices(*names):
    """인덱스 정리"""
    for name in names:
        try:
            es.indices.delete(index=name, ignore_unavailable=True)
        except Exception:
            pass

def wait_for_yellow(index, timeout=30):
    """인덱스가 yellow 이상이 될 때까지 대기"""
    es.cluster.health(index=index, wait_for_status="yellow", timeout=f"{timeout}s")

print("✅ 유틸리티 함수 로드 완료")

---
# Part 0: 검색의 진화 — Lexical의 한계와 Vector Search의 보완

### 🎯 의도
> BM25가 **정확한 키워드 매칭**에는 강하지만, **의미적 유사성**을 잡지 못하는 한계를 직접 체험한다.  
> Vector Search가 이를 어떻게 보완하는지, 그리고 왜 Hybrid가 필요한지를 이해한다.

In [ ]:
# 0-1. 테스트 인덱스 생성 + 샘플 데이터
cleanup_indices("lexical-vs-vector")

es.indices.create(
    index="lexical-vs-vector",
    body={
        "settings": {"number_of_shards": 1, "number_of_replicas": 0},
        "mappings": {
            "properties": {
                "title":   {"type": "text", "analyzer": "standard"},
                "content": {"type": "text", "analyzer": "standard"}
            }
        }
    }
)

docs = [
    {"_id": "1", "title": "API 엔드포인트 레이턴시 최적화",
     "content": "서비스 응답 시간이 급격히 증가하여 엔드포인트의 처리 지연을 줄이기 위한 캐시 전략과 비동기 처리를 도입했습니다."},
    {"_id": "2", "title": "API 응답 느림 현상 트러블슈팅",
     "content": "API 호출 시 응답이 느려지는 문제가 발생했습니다. 원인 분석 결과 데이터베이스 커넥션 풀 고갈이 확인되었습니다."},
    {"_id": "3", "title": "ERR-4521 에러 코드 분석",
     "content": "ERR-4521은 인증 토큰 만료 시 발생하는 에러입니다. 토큰 갱신 로직을 점검하세요."},
    {"_id": "4", "title": "자동차 연비 비교",
     "content": "2026년 출시된 전기차의 에너지 효율과 기존 내연기관 vehicle의 연료 소비를 비교합니다."},
    {"_id": "5", "title": "강아지 산책 가이드",
     "content": "공원에서 반려견과 함께 뛰어노는 방법과 안전 수칙을 정리했습니다. A puppy playing in the park needs proper supervision."},
]

actions = [{"_index": "lexical-vs-vector", "_id": d["_id"], "_source": {k:v for k,v in d.items() if k != "_id"}} for d in docs]
helpers.bulk(es, actions, refresh=True)
print(f"✅ lexical-vs-vector: {len(docs)}건 색인 완료")

In [ ]:
# 0-2. BM25의 한계 체험 — "API 응답 느림" 검색
# 기대: doc 2는 매칭되지만, doc 1 "엔드포인트 레이턴시"는 BM25로 매칭 불가

print("=" * 60)
print("🔍 Query DSL — BM25: 'API 응답 느림'")
print("=" * 60)
resp = es.search(
    index="lexical-vs-vector",
    body={
        "query": {"match": {"content": "API 응답 느림"}},
        "_source": ["title"]
    }
)
for hit in resp["hits"]["hits"]:
    print(f"  score={hit['_score']:.4f} | {hit['_source']['title']}")
if not resp["hits"]["hits"]:
    print("  (결과 없음)")

In [ ]:
# 0-2b. ES|QL — BM25 검색 (METADATA _score 필수!)
print("=" * 60)
print("🔍 ES|QL — BM25: 'API 응답 느림'")
print("=" * 60)
esql("""
FROM lexical-vs-vector METADATA _score
| WHERE MATCH(content, "API 응답 느림")
| KEEP title, _score
| SORT _score DESC
| LIMIT 5
""")

In [ ]:
# 0-3. 동의어/다국어 한계: "car fuel efficiency" 검색
# 기대: doc 4 "자동차"는 BM25로 매칭 불가

print("=" * 60)
print("🔍 Query DSL — BM25: 'car fuel efficiency' (다국어 한계)")
print("=" * 60)
resp = es.search(
    index="lexical-vs-vector",
    body={
        "query": {"match": {"content": "car fuel efficiency"}},
        "_source": ["title"]
    }
)
for hit in resp["hits"]["hits"]:
    print(f"  score={hit['_score']:.4f} | {hit['_source']['title']}")
if not resp["hits"]["hits"]:
    print("  (결과 없음) ← BM25는 'car' ≠ '자동차', 'vehicle'도 한글 문맥에서 매칭 안됨")

In [ ]:
# 0-4. 정확한 키워드는 BM25가 강함
print("=" * 60)
print("🔍 Query DSL — BM25: 'ERR-4521' (정확 매칭)")
print("=" * 60)
resp = es.search(
    index="lexical-vs-vector",
    body={
        "query": {"match": {"content": "ERR-4521"}},
        "_source": ["title"]
    }
)
for hit in resp["hits"]["hits"]:
    print(f"  score={hit['_score']:.4f} | {hit['_source']['title']}")

print()
print("=" * 60)
print("🔍 ES|QL — BM25: 'ERR-4521'")
print("=" * 60)
esql("""
FROM lexical-vs-vector METADATA _score
| WHERE MATCH(content, "ERR-4521")
| KEEP title, _score
| SORT _score DESC
| LIMIT 5
""")

### 0-5. Vector Search로 의미 검색 (Inference Endpoint 사용)

> 아래에서 `semantic_text` 필드를 사용하여 **동일 쿼리에서 BM25가 못잡는 의미적 매칭**을 확인합니다.  
> Inference Endpoint는 Part 2에서 상세히 다루며, 여기서는 먼저 생성만 합니다.

In [ ]:
# 0-5a. Inference Endpoint 생성 (jina-v5-text-small)
# 이미 존재하면 skip
try:
    es.inference.get(inference_id="jina-v5-for-poc", task_type="text_embedding")
    print("ℹ️ jina-v5-for-poc 이미 존재")
except Exception:
    es.inference.put(
        inference_id="jina-v5-for-poc",
        task_type="text_embedding",
        body={
            "service": "elastic",
            "service_settings": {
                "model_id": "jina-embeddings-v5-text-small",
                "adaptive_allocations": {"enabled": True, "min_number_of_allocations": 1}
            }
        }
    )
    print("✅ jina-v5-for-poc 생성 완료 (배포 대기 ~1분)")
    time.sleep(30)  # 모델 배포 대기

In [ ]:
# 0-5b. semantic_text 인덱스 생성 + 동일 데이터 색인
cleanup_indices("lexical-vs-vector-semantic")

es.indices.create(
    index="lexical-vs-vector-semantic",
    body={
        "mappings": {
            "properties": {
                "title":   {"type": "text"},
                "content": {
                    "type": "semantic_text",
                    "inference_id": "jina-v5-for-poc"
                }
            }
        }
    }
)

actions = [
    {"_index": "lexical-vs-vector-semantic", "_id": d["_id"],
     "_source": {k:v for k,v in d.items() if k != "_id"}}
    for d in docs
]
helpers.bulk(es, actions, refresh="wait_for")
print("✅ lexical-vs-vector-semantic: 색인 완료 (embedding 처리 중)")
time.sleep(10)  # embedding 완료 대기

In [ ]:
# 0-5c. Semantic Query: "API 응답이 느린 원인"
# 기대: doc 1 "엔드포인트 레이턴시"도 의미적으로 매칭!
print("=" * 60)
print("🔍 Query DSL — Semantic: 'API 응답이 느린 원인'")
print("=" * 60)
resp = es.search(
    index="lexical-vs-vector-semantic",
    body={
        "query": {
            "semantic": {
                "field": "content",
                "query": "API 응답이 느린 원인"
            }
        },
        "_source": ["title"]
    }
)
for hit in resp["hits"]["hits"]:
    print(f"  score={hit['_score']:.4f} | {hit['_source']['title']}")

In [ ]:
# 0-5d. 다국어 의미 검색: "car fuel efficiency" → "자동차" 매칭 확인
print("=" * 60)
print("🔍 Query DSL — Semantic: 'car fuel efficiency'")
print("=" * 60)
resp = es.search(
    index="lexical-vs-vector-semantic",
    body={
        "query": {
            "semantic": {
                "field": "content",
                "query": "car fuel efficiency"
            }
        },
        "_source": ["title"]
    }
)
for hit in resp["hits"]["hits"]:
    print(f"  score={hit['_score']:.4f} | {hit['_source']['title']}")

print()
print("📌 BM25에서 0건이었던 'car' → '자동차' 매칭이 Vector Search에서는 동작합니다!")

In [ ]:
# 0-5e. ES|QL — semantic_text 필드에 MATCH 사용
print("=" * 60)
print("🔍 ES|QL — Semantic: 'API 응답이 느린 원인'")
print("=" * 60)
esql("""
FROM lexical-vs-vector-semantic METADATA _score
| WHERE MATCH(content, "API 응답이 느린 원인")
| KEEP title, _score
| SORT _score DESC
| LIMIT 5
""")

---
# Part 1: 벡터 검색 내부 원리 — HNSW 파라미터 + ACORN

### 🎯 의도
> HNSW 핵심 파라미터(M, ef_construction, num_candidates)의 **실제 영향**을 확인한다.  
> Recall이 부족할 때 **num_candidates를 먼저 올려야 하는 이유**를 실습으로 증명한다.

In [ ]:
# 1-1. HNSW 파라미터 비교 인덱스 생성
DIMS = 128
cleanup_indices("hnsw-default", "hnsw-high")

# 기본값 (M=16, ef_construction=100)
es.indices.create(
    index="hnsw-default",
    body={
        "settings": {"number_of_shards": 1, "number_of_replicas": 0},
        "mappings": {"properties": {
            "category": {"type": "keyword"},
            "vec": {
                "type": "dense_vector", "dims": DIMS, "similarity": "cosine",
                "index_options": {"type": "hnsw", "m": 16, "ef_construction": 100}
            }
        }}
    }
)

# 고품질 (M=32, ef_construction=200)
es.indices.create(
    index="hnsw-high",
    body={
        "settings": {"number_of_shards": 1, "number_of_replicas": 0},
        "mappings": {"properties": {
            "category": {"type": "keyword"},
            "vec": {
                "type": "dense_vector", "dims": DIMS, "similarity": "cosine",
                "index_options": {"type": "hnsw", "m": 32, "ef_construction": 200}
            }
        }}
    }
)
print("✅ hnsw-default (M=16) + hnsw-high (M=32) 생성")

In [ ]:
# 1-2. 랜덤 벡터 데이터 10,000건 색인
NUM_DOCS = 10000
CATEGORIES = ["electronics", "clothing", "food", "books", "sports"]

def gen_normalized_vector(dims):
    vec = [random.gauss(0, 1) for _ in range(dims)]
    norm = math.sqrt(sum(v**2 for v in vec))
    return [round(v / norm, 6) for v in vec]

for idx_name in ["hnsw-default", "hnsw-high"]:
    actions = []
    for i in range(NUM_DOCS):
        vec = gen_normalized_vector(DIMS)
        actions.append({
            "_index": idx_name, "_id": str(i),
            "_source": {"category": random.choice(CATEGORIES), "vec": vec}
        })
    helpers.bulk(es, actions, refresh=True, chunk_size=1000)
    print(f"  ✅ {idx_name}: {NUM_DOCS}건 색인 완료")

print("✅ 모든 색인 완료")

In [ ]:
# 1-3. num_candidates 튜닝 효과 비교
# 동일 쿼리 벡터로 num_candidates 20 vs 500 비교

query_vec = gen_normalized_vector(DIMS)

print("=" * 60)
print("🔍 num_candidates=20 (낮음 — recall 저하 가능)")
print("=" * 60)
resp_low = es.search(
    index="hnsw-default",
    body={
        "size": 10,
        "knn": {"field": "vec", "query_vector": query_vec, "k": 10, "num_candidates": 20}
    }
)
ids_low = [h["_id"] for h in resp_low["hits"]["hits"]]
scores_low = [h["_score"] for h in resp_low["hits"]["hits"]]
print(f"  상위 10 IDs: {ids_low}")
print(f"  score range: {min(scores_low):.6f} ~ {max(scores_low):.6f}")

print()
print("=" * 60)
print("🔍 num_candidates=500 (높음 — recall 개선, 재인덱싱 불필요!)")
print("=" * 60)
resp_high = es.search(
    index="hnsw-default",
    body={
        "size": 10,
        "knn": {"field": "vec", "query_vector": query_vec, "k": 10, "num_candidates": 500}
    }
)
ids_high = [h["_id"] for h in resp_high["hits"]["hits"]]
scores_high = [h["_score"] for h in resp_high["hits"]["hits"]]
print(f"  상위 10 IDs: {ids_high}")
print(f"  score range: {min(scores_high):.6f} ~ {max(scores_high):.6f}")

overlap = len(set(ids_low) & set(ids_high))
print(f"\n📊 두 결과 겹침: {overlap}/10 → recall 차이 = {10 - overlap}건")
print("📌 핵심: num_candidates는 검색 시점에 조정 가능 → 재인덱싱 불필요!")

In [ ]:
# 1-4. Filter + kNN (ACORN 자동 적용, ES 9.1+)
print("=" * 60)
print("🔍 Filter + kNN: category='electronics'만 대상")
print("=" * 60)
resp = es.search(
    index="hnsw-default",
    body={
        "knn": {
            "field": "vec", "query_vector": query_vec,
            "k": 10, "num_candidates": 100,
            "filter": {"term": {"category": "electronics"}}
        },
        "_source": ["category"]
    }
)
cats = [h["_source"]["category"] for h in resp["hits"]["hits"]]
print(f"  반환 카테고리: {set(cats)} (all 'electronics' ✓)")
print(f"  결과 수: {len(resp['hits']['hits'])}")
print("📌 ES 9.1+: ACORN-1 자동 적용으로 filter+kNN 성능 최대 5× 개선")

In [ ]:
# 1-5. 인덱스 크기 비교: M=16 vs M=32
print("=" * 60)
print("📊 인덱스 크기 비교 (M=16 vs M=32)")
print("=" * 60)
for idx_name in ["hnsw-default", "hnsw-high"]:
    stats = es.indices.stats(index=idx_name)
    size_bytes = stats["indices"][idx_name]["primaries"]["store"]["size_in_bytes"]
    print(f"  {idx_name}: {size_bytes / 1024 / 1024:.2f} MB")

print("\n📌 M=32는 그래프 연결이 2배 → 인덱스 크기 증가하지만, 같은 num_candidates에서 더 높은 recall 가능")

---
# Part 2: ML 모델 분류 — Inference Endpoint · Bi-encoder · Cross-encoder

### 🎯 의도
> Elasticsearch Inference Endpoint를 통해 **Jina 시리즈 모델을 배포**하고,  
> Bi-encoder(검색용)와 Cross-encoder(Reranker)의 **역할 차이**를 실습으로 구분한다.

In [ ]:
# 2-1. Inference Endpoint 생성 — Jina Embeddings (이미 생성했으면 skip)
print("=" * 60)
print("📦 Inference Endpoint 상태 확인")
print("=" * 60)

# jina-v5 embedding (Part 0에서 생성)
try:
    ep = es.inference.get(inference_id="jina-v5-for-poc", task_type="text_embedding")
    print("✅ jina-v5-for-poc (text_embedding) — 존재")
except:
    print("⚠️ jina-v5-for-poc 없음 — Part 0 셀을 먼저 실행하세요")

In [ ]:
# 2-2. Inference Endpoint 생성 — Jina Reranker v3 (Cross-encoder)
try:
    es.inference.get(inference_id="jina-reranker-v3-poc", task_type="rerank")
    print("✅ jina-reranker-v3-poc (rerank) — 이미 존재")
except Exception:
    es.inference.put(
        inference_id="jina-reranker-v3-poc",
        task_type="rerank",
        body={
            "service": "elastic",
            "service_settings": {
                "model_id": "jina-reranker-v3",
                "adaptive_allocations": {"enabled": True, "min_number_of_allocations": 1}
            }
        }
    )
    print("✅ jina-reranker-v3-poc 생성 완료 (배포 대기 ~1분)")
    time.sleep(30)

In [ ]:
# 2-3. Bi-encoder 동작 확인 — 독립 인코딩
# 쿼리와 문서를 각각 독립적으로 임베딩 → 사전 인덱싱 가능
print("=" * 60)
print("🧠 Bi-encoder: 쿼리 임베딩")
print("=" * 60)
resp_q = es.inference.inference(
    inference_id="jina-v5-for-poc",
    task_type="text_embedding",
    body={"input": ["벡터 검색 최적화 방법"]}
)
emb_q = resp_q["text_embedding"][0]["embedding"]
print(f"  쿼리 벡터 차원: {len(emb_q)}d")
print(f"  처음 5값: {emb_q[:5]}")

print()
print("=" * 60)
print("🧠 Bi-encoder: 문서 임베딩 (독립 인코딩)")
print("=" * 60)
resp_d = es.inference.inference(
    inference_id="jina-v5-for-poc",
    task_type="text_embedding",
    body={"input": [
        "Elasticsearch에서 HNSW 파라미터를 조정하여 검색 성능을 향상시키는 방법",
        "오늘 점심 메뉴 추천",
        "vector search optimization techniques in distributed systems"
    ]}
)
for i, e in enumerate(resp_d["text_embedding"]):
    print(f"  문서 {i+1}: {len(e['embedding'])}d | 처음 3값: {e['embedding'][:3]}")

print("\n📌 Bi-encoder: 쿼리·문서를 독립 인코딩 → 사전 인덱싱 가능 (수백만 건 처리)")

In [ ]:
# 2-4. Cross-encoder(Reranker) 동작 확인 — 쌍 비교
# 쿼리+문서를 동시 입력 → 양방향 attention으로 정밀 scoring
print("=" * 60)
print("🎯 Cross-encoder (Reranker): 쿼리+문서 동시 평가")
print("=" * 60)

resp_r = es.inference.inference(
    inference_id="jina-reranker-v3-poc",
    task_type="rerank",
    body={
        "input": [
            "Elasticsearch에서 HNSW 파라미터를 조정하여 검색 성능을 향상시키는 방법",
            "오늘 점심 메뉴 추천",
            "vector search optimization techniques in distributed systems",
            "API 엔드포인트의 응답 레이턴시를 줄이기 위한 캐시 전략"
        ],
        "query": "벡터 검색 최적화 방법"
    }
)

for r in sorted(resp_r["rerank"], key=lambda x: x["relevance_score"], reverse=True):
    idx = r["index"]
    score = r["relevance_score"]
    texts = [
        "HNSW 파라미터 조정으로 검색 성능 향상",
        "오늘 점심 메뉴 추천",
        "vector search optimization techniques",
        "API 레이턴시 줄이기 위한 캐시 전략"
    ]
    print(f"  score={score:.6f} | {texts[idx]}")

print("\n📌 Cross-encoder: 쿼리와 각 문서를 쌍으로 평가 → Bi-encoder보다 정밀, 하지만 사전 인덱싱 불가")
print("📌 따라서 top-K만 재순위하는 2단계 구조 필수: Bi-encoder(top-100) → Cross-encoder(top-10)")

---
# Part 3: Chunking + semantic_text vs dense_vector

### 🎯 의도
> `semantic_text`의 **자동 chunking + embedding 파이프라인**을 이해하고,  
> `dense_vector`로 **세밀하게 제어**하는 방식과의 차이를 비교한다.

In [ ]:
# 3-1. semantic_text 인덱스 (자동 파이프라인)
cleanup_indices("semantic-auto", "vector-manual")

es.indices.create(
    index="semantic-auto",
    body={
        "mappings": {"properties": {
            "title":   {"type": "text"},
            "content": {"type": "semantic_text", "inference_id": "jina-v5-for-poc"}
        }}
    }
)

# 긴 문서 색인 → 자동 chunking 확인
long_doc = (
    "Elasticsearch는 Apache Lucene 기반의 분산 검색 엔진입니다. "
    "버전 8.0부터 네이티브 벡터 검색을 지원하며, HNSW 알고리즘을 사용합니다. "
    "HNSW는 Hierarchical Navigable Small World의 약자로, 다층 그래프 구조를 통해 "
    "O(log N) 시간 복잡도로 근사 최근접 이웃을 찾습니다. "
    "핵심 파라미터는 M(노드당 최대 연결 수), ef_construction(구축 시 탐색 범위), "
    "num_candidates(검색 시 후보 수)입니다. "
    "버전 9.1부터는 BBQ(Better Binary Quantization)가 384차원 이상 벡터의 기본 양자화 방식으로 "
    "자동 적용되어, float32 대비 약 32배 압축과 95% 메모리 절감을 달성합니다. "
    "Hybrid Search는 BM25 lexical 검색과 벡터 검색을 결합하여 두 방식의 장점을 모두 활용합니다. "
    "RRF(Reciprocal Rank Fusion)는 가장 간단한 결합 방법이며, Linear Retriever는 score 차이를 정확히 반영합니다. "
    "Reranker는 Cross-encoder를 사용하여 최종 결과의 정밀도를 극대화합니다. "
    "RAG 파이프라인에서 검색 품질은 LLM 출력 품질을 직접 결정합니다."
)

es.index(index="semantic-auto", id="1", body={"title": "Elasticsearch 벡터 검색 가이드", "content": long_doc}, refresh="wait_for")
time.sleep(10)  # embedding 완료 대기
print("✅ semantic-auto: 긴 문서 1건 색인 완료")

In [ ]:
# 3-2. 자동 chunking 결과 확인
print("=" * 60)
print("📋 semantic_text 내부 chunk 구조 확인")
print("=" * 60)

doc = es.get(index="semantic-auto", id="1")
content_field = doc["_source"]["content"]

# semantic_text는 내부적으로 chunks를 nested로 저장
if isinstance(content_field, dict) and "chunks" in content_field:
    chunks = content_field["chunks"]
    print(f"총 chunk 수: {len(chunks)}")
    for i, chunk in enumerate(chunks):
        text = chunk.get("text", "")[:80]
        emb_len = len(chunk.get("embedding", []))
        print(f"  chunk[{i}]: {emb_len}d | '{text}...'")
elif isinstance(content_field, str):
    print("(embedding 처리 중이거나 raw text 반환 — 잠시 후 재시도)")
else:
    print(f"필드 구조: {type(content_field)}")
    print(json.dumps(content_field, ensure_ascii=False, indent=2)[:500])

In [ ]:
# 3-3. semantic_text로 검색
print("=" * 60)
print("🔍 Query DSL — semantic query: '벡터 압축 기술'")
print("=" * 60)
resp = es.search(
    index="semantic-auto",
    body={
        "query": {"semantic": {"field": "content", "query": "벡터 압축 기술"}},
        "_source": ["title"]
    }
)
for hit in resp["hits"]["hits"]:
    print(f"  score={hit['_score']:.4f} | {hit['_source']['title']}")

print()
print("=" * 60)
print("🔍 ES|QL — semantic query: '벡터 압축 기술'")
print("=" * 60)
esql("""
FROM semantic-auto METADATA _score
| WHERE MATCH(content, "벡터 압축 기술")
| KEEP title, _score
| SORT _score DESC
| LIMIT 5
""")

In [ ]:
# 3-4. dense_vector 인덱스 (수동 제어) — bbq_hnsw
es.indices.create(
    index="vector-manual",
    body={
        "settings": {"number_of_shards": 1, "number_of_replicas": 0},
        "mappings": {"properties": {
            "title":    {"type": "text"},
            "content":  {"type": "text"},
            "chunk_id": {"type": "keyword"},
            "content_vector": {
                "type": "dense_vector", "dims": 1024, "similarity": "cosine",
                "index_options": {"type": "bbq_hnsw"}
            }
        }}
    }
)

# 수동으로 embedding 생성 + 색인
resp_emb = es.inference.inference(
    inference_id="jina-v5-for-poc",
    task_type="text_embedding",
    body={"input": [long_doc[:500]]}  # 첫 500자를 하나의 chunk로
)
vec = resp_emb["text_embedding"][0]["embedding"]

es.index(
    index="vector-manual", id="1",
    body={
        "title": "ES 벡터 검색 가이드",
        "content": long_doc[:500],
        "chunk_id": "chunk-001",
        "content_vector": vec
    },
    refresh=True
)
print(f"✅ vector-manual: 1건 색인 (수동 {len(vec)}d embedding)")

In [ ]:
# 3-5. dense_vector kNN 검색
query_emb = es.inference.inference(
    inference_id="jina-v5-for-poc", task_type="text_embedding",
    body={"input": ["벡터 압축 기술"]}
)
qvec = query_emb["text_embedding"][0]["embedding"]

print("=" * 60)
print("🔍 Query DSL — kNN: '벡터 압축 기술'")
print("=" * 60)
resp = es.search(
    index="vector-manual",
    body={
        "knn": {"field": "content_vector", "query_vector": qvec, "k": 5, "num_candidates": 50},
        "_source": ["title", "chunk_id"]
    }
)
for hit in resp["hits"]["hits"]:
    print(f"  score={hit['_score']:.6f} | {hit['_source']['title']} ({hit['_source']['chunk_id']})")

print()
print("📌 semantic_text는 빠른 시작에 최적 → 한계 시 dense_vector로 전환")

---
# Part 4: 양자화 — BBQ · DiskBBQ

### 🎯 의도
> BBQ의 **32배 압축 원리**를 체감하고, float32 대비 **인덱스 크기와 recall**을 직접 비교한다.

In [ ]:
# 4-1. float32 vs BBQ vs DiskBBQ 인덱스 생성
QDIMS = 1024
cleanup_indices("vectors-float32", "vectors-bbq", "vectors-diskbbq")

configs = {
    "vectors-float32": {"type": "hnsw", "m": 16, "ef_construction": 100},
    "vectors-bbq":     {"type": "bbq_hnsw"},
    "vectors-diskbbq": {"type": "bbq_disk"},
}

for name, opts in configs.items():
    es.indices.create(
        index=name,
        body={
            "settings": {"number_of_shards": 1, "number_of_replicas": 0},
            "mappings": {"properties": {
                "vec": {"type": "dense_vector", "dims": QDIMS, "similarity": "cosine",
                        "index_options": opts}
            }}
        }
    )
    print(f"  ✅ {name} ({opts['type']}) 생성")

In [ ]:
# 4-2. 50,000건 색인 (3개 인덱스 동시)
NUM_QDOCS = 50000

# 벡터를 한 번 생성하여 3개 인덱스에 동일하게 색인
print("🔄 50,000건 벡터 생성 + 색인 중...")
for idx_name in configs.keys():
    actions = []
    random.seed(42)  # 동일 벡터 보장
    for i in range(NUM_QDOCS):
        vec = gen_normalized_vector(QDIMS)
        actions.append({"_index": idx_name, "_id": str(i), "_source": {"vec": vec}})
        if len(actions) >= 2000:
            helpers.bulk(es, actions, refresh=False, chunk_size=500)
            actions = []
    if actions:
        helpers.bulk(es, actions, refresh=False, chunk_size=500)
    es.indices.refresh(index=idx_name)
    print(f"  ✅ {idx_name}: {NUM_QDOCS}건 완료")

print("✅ 전체 색인 완료")

In [ ]:
# 4-3. 인덱스 크기 비교
print("=" * 60)
print(f"📊 인덱스 크기 비교 ({QDIMS}d × {NUM_QDOCS:,}건)")
print("=" * 60)
print(f"  {'인덱스':<20} {'크기':>10} {'압축률':>8}")
print(f"  {'-'*40}")

sizes = {}
for idx_name in configs.keys():
    stats = es.indices.stats(index=idx_name)
    size_bytes = stats["indices"][idx_name]["primaries"]["store"]["size_in_bytes"]
    sizes[idx_name] = size_bytes
    
base = sizes["vectors-float32"]
for idx_name, size_bytes in sizes.items():
    mb = size_bytes / 1024 / 1024
    ratio = base / size_bytes if size_bytes > 0 else 0
    print(f"  {idx_name:<20} {mb:>8.1f}MB {ratio:>7.1f}×")

print(f"\n📌 BBQ: float32 대비 ~{base / sizes['vectors-bbq']:.0f}× 압축 (이론치 ~32×)")
print("📌 ES 9.1+에서 384d 이상이면 BBQ가 기본값 → 설정 불필요")

In [ ]:
# 4-4. Recall 비교 — float32 vs BBQ
random.seed(99)
qvec = gen_normalized_vector(QDIMS)

results = {}
for idx_name in ["vectors-float32", "vectors-bbq", "vectors-diskbbq"]:
    resp = es.search(
        index=idx_name,
        body={"size": 10, "knn": {"field": "vec", "query_vector": qvec, "k": 10, "num_candidates": 100}}
    )
    results[idx_name] = [h["_id"] for h in resp["hits"]["hits"]]

print("=" * 60)
print("📊 Recall 비교 (float32 기준)")
print("=" * 60)
gt = set(results["vectors-float32"])
for idx_name, ids in results.items():
    overlap = len(gt & set(ids))
    print(f"  {idx_name:<20} | 겹침: {overlap}/10 | recall: {overlap*10}%")

print("\n📌 BBQ가 float32보다 ranking이 좋을 수 있음 (더 넓은 탐색 + oversampling rescore)")

---
# Part 5: Hybrid Search — RRF · Linear · Reranker + ES|QL FORK/FUSE/RERANK

### 🎯 의도
> Query DSL Retriever 프레임워크와 ES|QL `FORK`/`FUSE`/`RERANK` 모두를 사용하여  
> **BM25 + kNN + RRF/Linear + Reranker** 파이프라인을 구성한다.

In [ ]:
# 5-1. Hybrid 검색용 인덱스 준비
cleanup_indices("hybrid-demo")

es.indices.create(
    index="hybrid-demo",
    body={
        "settings": {"number_of_shards": 1, "number_of_replicas": 0},
        "mappings": {"properties": {
            "title":   {"type": "text", "analyzer": "standard"},
            "content": {"type": "text", "analyzer": "standard"},
            "content_semantic": {"type": "semantic_text", "inference_id": "jina-v5-for-poc"},
            "category": {"type": "keyword"}
        }}
    }
)

hybrid_docs = [
    {"_id": "1", "title": "Elasticsearch HNSW 튜닝 가이드",
     "content": "Elasticsearch의 HNSW 알고리즘 파라미터를 조정하여 벡터 검색 성능을 최적화하는 방법을 설명합니다.",
     "content_semantic": "Elasticsearch의 HNSW 알고리즘 파라미터를 조정하여 벡터 검색 성능을 최적화하는 방법을 설명합니다.",
     "category": "search"},
    {"_id": "2", "title": "ERR-4521 인증 토큰 만료 해결",
     "content": "ERR-4521 에러 코드는 인증 토큰이 만료되었을 때 발생합니다. 자동 갱신 로직을 구현하세요.",
     "content_semantic": "ERR-4521 에러 코드는 인증 토큰이 만료되었을 때 발생합니다. 자동 갱신 로직을 구현하세요.",
     "category": "troubleshooting"},
    {"_id": "3", "title": "API 레이턴시 개선 사례",
     "content": "마이크로서비스 아키텍처에서 API 응답 시간을 줄이기 위해 캐시 레이어와 비동기 처리를 도입한 사례입니다.",
     "content_semantic": "마이크로서비스 아키텍처에서 API 응답 시간을 줄이기 위해 캐시 레이어와 비동기 처리를 도입한 사례입니다.",
     "category": "performance"},
    {"_id": "4", "title": "벡터 양자화 BBQ 원리",
     "content": "BBQ는 float32 벡터를 1-bit로 양자화하면서 corrective factors로 정보 손실을 보정합니다. 32배 압축을 달성합니다.",
     "content_semantic": "BBQ는 float32 벡터를 1-bit로 양자화하면서 corrective factors로 정보 손실을 보정합니다. 32배 압축을 달성합니다.",
     "category": "search"},
    {"_id": "5", "title": "검색 성능 벤치마크 도구",
     "content": "Rally를 사용하여 Elasticsearch 클러스터의 검색 성능을 측정하고 병목을 분석하는 방법입니다.",
     "content_semantic": "Rally를 사용하여 Elasticsearch 클러스터의 검색 성능을 측정하고 병목을 분석하는 방법입니다.",
     "category": "performance"},
]

actions = [{"_index": "hybrid-demo", "_id": d["_id"], "_source": {k:v for k,v in d.items() if k != "_id"}} for d in hybrid_docs]
helpers.bulk(es, actions, refresh="wait_for")
time.sleep(10)  # embedding 대기
print(f"✅ hybrid-demo: {len(hybrid_docs)}건 색인 완료")

In [ ]:
# 5-2. Query DSL — RRF Hybrid Search (기본)
print("=" * 60)
print("🔍 Query DSL — RRF: BM25 + Semantic")
print("=" * 60)

resp = es.search(
    index="hybrid-demo",
    body={
        "retriever": {
            "rrf": {
                "retrievers": [
                    {"standard": {"query": {"match": {"content": "벡터 검색 성능 최적화"}}}},
                    {"standard": {"query": {"semantic": {"field": "content_semantic", "query": "벡터 검색 성능 최적화"}}}}
                ]
            }
        },
        "_source": ["title"]
    }
)
for hit in resp["hits"]["hits"]:
    print(f"  score={hit['_score']:.6f} | {hit['_source']['title']}")

In [ ]:
# 5-3. ES|QL — FORK + FUSE (RRF) Hybrid Search
# ES 9.1+ tech preview
print("=" * 60)
print("🔍 ES|QL — FORK + FUSE (RRF): BM25 + Semantic")
print("=" * 60)

esql("""
FROM hybrid-demo METADATA _score
| FORK
    ( WHERE MATCH(content, "벡터 검색 성능 최적화") | SORT _score DESC | LIMIT 10 ),
    ( WHERE MATCH(content_semantic, "벡터 검색 성능 최적화") | SORT _score DESC | LIMIT 10 )
| FUSE { "method": "rrf" }
| SORT _score DESC
| LIMIT 5
| KEEP title, _score
""")

In [ ]:
# 5-4. ES|QL — FORK + FUSE (Linear with MinMax normalization)
print("=" * 60)
print("🔍 ES|QL — FORK + FUSE (Linear + MinMax): BM25 + Semantic")
print("=" * 60)

esql("""
FROM hybrid-demo METADATA _score
| FORK
    ( WHERE MATCH(content, "벡터 검색 성능 최적화") | SORT _score DESC | LIMIT 10 ),
    ( WHERE MATCH(content_semantic, "벡터 검색 성능 최적화") | SORT _score DESC | LIMIT 10 )
| FUSE { "method": "linear", "weights": [0.3, 0.7], "normalizer": "minmax" }
| SORT _score DESC
| LIMIT 5
| KEEP title, _score
""")

In [ ]:
# 5-5. Query DSL — Weighted RRF (retriever별 가중치)
print("=" * 60)
print("🔍 Query DSL — Weighted RRF: BM25(0.3) + Semantic(0.7)")
print("=" * 60)

resp = es.search(
    index="hybrid-demo",
    body={
        "retriever": {
            "rrf": {
                "retrievers": [
                    {"standard": {"query": {"match": {"content": "벡터 검색 성능 최적화"}}}},
                    {"standard": {"query": {"semantic": {"field": "content_semantic", "query": "벡터 검색 성능 최적화"}}}}
                ],
                "retriever_weights": [0.3, 0.7]
            }
        },
        "_source": ["title"]
    }
)
for hit in resp["hits"]["hits"]:
    print(f"  score={hit['_score']:.6f} | {hit['_source']['title']}")

In [ ]:
# 5-6. Query DSL — Linear Retriever (score 차이 반영)
print("=" * 60)
print("🔍 Query DSL — Linear: BM25(0.3) + Semantic(0.7)")
print("=" * 60)

resp = es.search(
    index="hybrid-demo",
    body={
        "retriever": {
            "linear": {
                "retrievers": [
                    {"retriever": {"standard": {"query": {"match": {"content": "벡터 검색 성능 최적화"}}}}, "weight": 0.3},
                    {"retriever": {"standard": {"query": {"semantic": {"field": "content_semantic", "query": "벡터 검색 성능 최적화"}}}}, "weight": 0.7}
                ]
            }
        },
        "_source": ["title"]
    }
)
for hit in resp["hits"]["hits"]:
    print(f"  score={hit['_score']:.6f} | {hit['_source']['title']}")

In [ ]:
# 5-7. Query DSL — 3단계 파이프라인: BM25 + kNN + RRF + Reranker
print("=" * 60)
print("🔍 Query DSL — 3단계: RRF + Reranker")
print("=" * 60)

resp = es.search(
    index="hybrid-demo",
    body={
        "retriever": {
            "text_similarity_reranker": {
                "retriever": {
                    "rrf": {
                        "retrievers": [
                            {"standard": {"query": {"match": {"content": "벡터 검색 최적화"}}}},
                            {"standard": {"query": {"semantic": {"field": "content_semantic", "query": "벡터 검색 최적화"}}}}
                        ]
                    }
                },
                "field": "content",
                "inference_id": "jina-reranker-v3-poc",
                "inference_text": "벡터 검색 최적화",
                "rank_window_size": 20,
                "min_score": 0.01
            }
        },
        "_source": ["title"]
    }
)
for hit in resp["hits"]["hits"]:
    print(f"  score={hit['_score']:.6f} | {hit['_source']['title']}")

print("\n📌 단일 _search 호출로 BM25 + Semantic + RRF + Reranking 전체 파이프라인 실행!")

In [ ]:
# 5-8. ES|QL — FORK + FUSE + RERANK (전체 파이프라인)
# RERANK는 ES 9.2+ tech preview
print("=" * 60)
print("🔍 ES|QL — FORK + FUSE + RERANK (3단계 파이프라인)")
print("=" * 60)

try:
    esql("""
FROM hybrid-demo METADATA _score
| FORK
    ( WHERE MATCH(content, "벡터 검색 최적화") | SORT _score DESC | LIMIT 20 ),
    ( WHERE MATCH(content_semantic, "벡터 검색 최적화") | SORT _score DESC | LIMIT 20 )
| FUSE { "method": "rrf" }
| SORT _score DESC
| LIMIT 10
| RERANK "벡터 검색 최적화" ON content WITH { "inference_id": "jina-reranker-v3-poc" }
| LIMIT 5
| KEEP title, _score
""")
except Exception as e:
    print(f"⚠️ RERANK 에러 (9.2+ tech preview 필요): {e}")
    print("   → 클러스터 버전이 9.2 미만이면 Query DSL retriever를 사용하세요")

In [ ]:
# 5-9. ES|QL — RERANK에서 여러 필드 결합
# RERANK의 강점: 매핑된 필드가 아닌 expression도 가능
print("=" * 60)
print("🔍 ES|QL — RERANK: 여러 필드 결합 (title + content)")
print("=" * 60)

try:
    esql("""
FROM hybrid-demo METADATA _score
| WHERE MATCH(content, "벡터 검색 최적화")
| SORT _score DESC
| LIMIT 20
| RERANK "벡터 검색 최적화" ON CONCAT(title, " ", content) WITH { "inference_id": "jina-reranker-v3-poc" }
| LIMIT 5
| KEEP title, _score
""")
except Exception as e:
    print(f"⚠️ RERANK 에러: {e}")

---
# Part 6: RAG 실전 설계 — Context Engineering

### 🎯 의도
> RAG 파이프라인에서 **검색 품질이 LLM 출력을 결정**한다는 원칙을 실습으로 확인한다.  
> `min_score` cutoff, context 길이 제어, self-contained chunk 기법을 적용한다.

In [ ]:
# 6-1. RAG용 인덱스 — self-contained chunks (문서 요약 삽입)
cleanup_indices("rag-demo")

es.indices.create(
    index="rag-demo",
    body={
        "mappings": {"properties": {
            "doc_title":   {"type": "text"},
            "content":     {"type": "text"},
            "content_semantic": {"type": "semantic_text", "inference_id": "jina-v5-for-poc"},
            "category":    {"type": "keyword"},
            "doc_summary": {"type": "text"}
        }}
    }
)

rag_docs = [
    {"_id": "1", "doc_title": "ES 벡터 검색 가이드", "category": "search",
     "doc_summary": "이 문서는 Elasticsearch 벡터 검색의 원리와 최적화를 다룹니다.",
     "content": "[요약] ES 벡터 검색 원리와 최적화 문서.\nHNSW는 다층 그래프 구조로 O(log N) 시간에 근사 최근접 이웃을 찾습니다. 핵심 파라미터는 M, ef_construction, num_candidates입니다.",
     "content_semantic": "[요약] ES 벡터 검색 원리와 최적화 문서.\nHNSW는 다층 그래프 구조로 O(log N) 시간에 근사 최근접 이웃을 찾습니다. 핵심 파라미터는 M, ef_construction, num_candidates입니다."},
    {"_id": "2", "doc_title": "ES 벡터 검색 가이드", "category": "search",
     "doc_summary": "이 문서는 Elasticsearch 벡터 검색의 원리와 최적화를 다룹니다.",
     "content": "[요약] ES 벡터 검색 원리와 최적화 문서.\nBBQ는 1-bit 양자화 + corrective factors로 32배 압축하면서 recall을 유지합니다. ES 9.1+에서 384d 이상 자동 적용됩니다.",
     "content_semantic": "[요약] ES 벡터 검색 원리와 최적화 문서.\nBBQ는 1-bit 양자화 + corrective factors로 32배 압축하면서 recall을 유지합니다. ES 9.1+에서 384d 이상 자동 적용됩니다."},
    {"_id": "3", "doc_title": "API 트러블슈팅 매뉴얼", "category": "troubleshooting",
     "doc_summary": "API 관련 에러 코드와 해결 방법을 정리한 매뉴얼입니다.",
     "content": "[요약] API 에러 코드와 해결 매뉴얼.\nERR-4521은 인증 토큰 만료 에러입니다. 토큰 자동 갱신 로직을 구현하면 해결됩니다.",
     "content_semantic": "[요약] API 에러 코드와 해결 매뉴얼.\nERR-4521은 인증 토큰 만료 에러입니다. 토큰 자동 갱신 로직을 구현하면 해결됩니다."},
    {"_id": "4", "doc_title": "사내 카페 메뉴", "category": "general",
     "doc_summary": "구내식당 카페 메뉴와 가격표입니다.",
     "content": "[요약] 구내식당 카페 메뉴.\n아메리카노 2500원, 카페라떼 3000원, 녹차라떼 3500원.",
     "content_semantic": "[요약] 구내식당 카페 메뉴.\n아메리카노 2500원, 카페라떼 3000원, 녹차라떼 3500원."},
]

actions = [{"_index": "rag-demo", "_id": d["_id"], "_source": {k:v for k,v in d.items() if k != "_id"}} for d in rag_docs]
helpers.bulk(es, actions, refresh="wait_for")
time.sleep(10)
print(f"✅ rag-demo: {len(rag_docs)}건 색인 완료")

In [ ]:
# 6-2. Hybrid + Reranker + min_score cutoff
print("=" * 60)
print("🔍 RAG 파이프라인: Hybrid + Reranker + min_score")
print("=" * 60)

resp = es.search(
    index="rag-demo",
    body={
        "size": 3,
        "retriever": {
            "text_similarity_reranker": {
                "retriever": {
                    "rrf": {
                        "retrievers": [
                            {"standard": {"query": {"match": {"content": "벡터 압축 양자화 방법"}}}},
                            {"standard": {"query": {"semantic": {"field": "content_semantic", "query": "벡터 압축 양자화 방법"}}}}
                        ]
                    }
                },
                "field": "content",
                "inference_id": "jina-reranker-v3-poc",
                "inference_text": "벡터 압축 양자화 방법",
                "rank_window_size": 20,
                "min_score": 0.1
            }
        },
        "_source": ["doc_title", "content", "category"]
    }
)

for hit in resp["hits"]["hits"]:
    s = hit["_source"]
    print(f"  score={hit['_score']:.4f} | [{s['category']}] {s['doc_title']}")
    print(f"    → {s['content'][:60]}...")

filtered = [h for h in resp["hits"]["hits"]]
print(f"\n📌 min_score=0.1 → '사내 카페 메뉴' 같은 무관 문서 필터링됨")
print(f"📌 이것이 RAG hallucination 방지의 첫 번째 방어선!")

In [ ]:
# 6-3. ES|QL — FORK + FUSE + RERANK (RAG 파이프라인)
print("=" * 60)
print("🔍 ES|QL — RAG용 Hybrid + RERANK")
print("=" * 60)

try:
    esql("""
FROM rag-demo METADATA _score
| FORK
    ( WHERE MATCH(content, "벡터 압축 양자화") | SORT _score DESC | LIMIT 10 ),
    ( WHERE MATCH(content_semantic, "벡터 압축 양자화") | SORT _score DESC | LIMIT 10 )
| FUSE { "method": "rrf" }
| SORT _score DESC
| LIMIT 10
| RERANK "벡터 압축 양자화 방법" ON content WITH { "inference_id": "jina-reranker-v3-poc" }
| WHERE _score > 0.1
| LIMIT 3
| KEEP doc_title, category, _score
""")
except Exception as e:
    print(f"⚠️ ES|QL 에러: {e}")
    print("   → 9.2+ 필요 또는 RERANK tech preview 미활성화")

In [ ]:
# 6-4. LLM에 전달할 Context 조립 함수
def search_for_rag(query, top_k=3, min_score=0.1):
    """Hybrid + Reranker 검색 → LLM context 조립"""
    resp = es.search(
        index="rag-demo",
        body={
            "size": top_k,
            "retriever": {
                "text_similarity_reranker": {
                    "retriever": {
                        "rrf": {
                            "retrievers": [
                                {"standard": {"query": {"match": {"content": query}}}},
                                {"standard": {"query": {"semantic": {"field": "content_semantic", "query": query}}}}
                            ]
                        }
                    },
                    "field": "content",
                    "inference_id": "jina-reranker-v3-poc",
                    "inference_text": query,
                    "rank_window_size": 20,
                    "min_score": min_score
                }
            },
            "_source": ["doc_title", "content"]
        }
    )
    
    contexts = []
    for hit in resp["hits"]["hits"]:
        src = hit["_source"]
        contexts.append(f"[출처: {src['doc_title']}] (score: {hit['_score']:.4f})\n{src['content']}")
    
    context_block = "\n---\n".join(contexts)
    
    prompt = f"""다음 검색 결과를 바탕으로 질문에 답변하세요. 
검색 결과에 없는 내용은 "정보가 없습니다"라고 답하세요.

## 검색 결과
{context_block}

## 질문
{query}

## 답변"""
    
    return prompt, len(context_block.split()), resp["hits"]["hits"]


# 사용 예시
query = "BBQ 양자화의 압축률은?"
prompt, word_count, hits = search_for_rag(query)

print("=" * 60)
print(f"📋 RAG Context 조립 결과")
print("=" * 60)
print(f"  검색된 chunk 수: {len(hits)}")
print(f"  Context 길이: {word_count} words")
print(f"  8K 토큰 제한 이내: {'✅' if word_count < 6000 else '❌'}")
print()
print("--- LLM Prompt Preview ---")
print(prompt[:500])
print("...")

In [ ]:
# 6-5. 검색 품질 평가 — Ranking Evaluation API
print("=" * 60)
print("📊 검색 품질 정량 평가 (nDCG@10)")
print("=" * 60)

resp = es.rank_eval(
    index="rag-demo",
    body={
        "requests": [
            {
                "id": "query_bbq",
                "request": {
                    "query": {"semantic": {"field": "content_semantic", "query": "벡터 양자화 압축"}}
                },
                "ratings": [
                    {"_index": "rag-demo", "_id": "2", "rating": 3},
                    {"_index": "rag-demo", "_id": "1", "rating": 2},
                    {"_index": "rag-demo", "_id": "3", "rating": 0},
                    {"_index": "rag-demo", "_id": "4", "rating": 0},
                ]
            },
            {
                "id": "query_hnsw",
                "request": {
                    "query": {"semantic": {"field": "content_semantic", "query": "HNSW 파라미터"}}
                },
                "ratings": [
                    {"_index": "rag-demo", "_id": "1", "rating": 3},
                    {"_index": "rag-demo", "_id": "2", "rating": 1},
                    {"_index": "rag-demo", "_id": "3", "rating": 0},
                ]
            }
        ],
        "metric": {"dcg": {"k": 10, "normalize": True}}
    }
)

print(f"  전체 nDCG@10: {resp['metric_score']:.4f}")
for qid, detail in resp["details"].items():
    print(f"  {qid}: nDCG={detail['metric_score']:.4f}")

print("\n📌 _rank_eval로 검색 품질의 정량 지표를 확보 → Reranker/chunking 변경 전후 객관 비교 가능")

In [ ]:
# 6-6. ES|QL — 인덱스 데이터 분포 확인
print("=" * 60)
print("📊 ES|QL — rag-demo 카테고리 분포")
print("=" * 60)
esql("""
FROM rag-demo
| STATS doc_count = COUNT(*) BY category
| SORT doc_count DESC
""")

---
# 🧹 전체 Cleanup

> PoC 완료 후 테스트 인덱스와 Inference Endpoint를 정리합니다.

In [ ]:
# 전체 인덱스 정리
indices_to_clean = [
    "lexical-vs-vector", "lexical-vs-vector-semantic",
    "hnsw-default", "hnsw-high",
    "semantic-auto", "vector-manual",
    "vectors-float32", "vectors-bbq", "vectors-diskbbq",
    "hybrid-demo", "rag-demo"
]

for idx in indices_to_clean:
    try:
        es.indices.delete(index=idx, ignore_unavailable=True)
        print(f"  🗑️ {idx} 삭제")
    except Exception as e:
        print(f"  ⚠️ {idx}: {e}")

# Inference Endpoint 정리
for ep_id, task in [("jina-v5-for-poc", "text_embedding"), ("jina-reranker-v3-poc", "rerank")]:
    try:
        es.inference.delete(inference_id=ep_id, task_type=task, force=True)
        print(f"  🗑️ {ep_id} 삭제")
    except Exception as e:
        print(f"  ⚠️ {ep_id}: {e}")

print("\n✅ 전체 정리 완료")

---
# 📋 교육 주제 ↔ 실습 매핑 요약

| 슬라이드 주제 | PoC 섹션 | Query DSL | ES\|QL | 핵심 확인 사항 |
|---------------|----------|-----------|--------|---------------|
| Lexical → Vector → Hybrid 진화 | Part 0 | ✅ match, semantic | ✅ MATCH + METADATA _score | BM25 한계 vs semantic 보완 |
| HNSW 내부 원리 + ACORN | Part 1 | ✅ knn, filter | ✅ 통계 쿼리 | num_candidates 튜닝 효과 |
| Bi-encoder vs Cross-encoder | Part 2 | ✅ _inference API | — | 독립 인코딩 vs 쌍 비교 |
| Chunking + semantic_text | Part 3 | ✅ semantic, knn | ✅ MATCH on semantic_text | 자동 chunking 결과, _explain |
| BBQ 32배 압축 원리 | Part 4 | ✅ knn 비교 | — | 인덱스 크기 비교, recall 비교 |
| Hybrid (RRF/Linear/Reranker) | Part 5 | ✅ rrf, linear, reranker | ✅ FORK/FUSE/RERANK | 3단계 파이프라인, score 추적 |
| RAG 실전 설계 | Part 6 | ✅ reranker + min_score | ✅ FORK/FUSE/RERANK | Context Engineering, _rank_eval |

---

> **ES\|QL 주요 문법 참고**
> - `METADATA _score` — `_score` 컬럼 사용 시 **반드시 FROM 뒤에** 명시
> - `MATCH(field, query)` — `text`, `keyword`, `semantic_text` 필드 모두 지원
> - `KNN(field, dims, query_vector)` — dense_vector 필드 kNN 검색 (9.2+)
> - `FORK (...), (...)` — 다중 검색 branch 실행 (9.1+ tech preview)
> - `FUSE { "method": "rrf" | "linear" }` — FORK 결과 결합 (9.1+ tech preview)
> - `RERANK query ON field(s) WITH { "inference_id": "..." }` — 재순위 (9.2+ tech preview)
> - `COMPLETION prompt WITH { "inference_id": "..." }` — LLM 호출 (9.2+ tech preview)

---
> **문서 작성**: Claude (Apr 2026) | **교육 자료**: Vector Search Deep Dive Part A + B 기반 | **환경**: ES 9.x (ECH)